# Structured outputs
- LLM 응답을 서비스 코드에서 사용하려면  자유 텍스트보다  구조화된 json이 훨씬 다루기 쉽다.

In [4]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델 명을 읽어온다
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY를 환경변수로 설정하세요")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL","gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 :", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 : gpt-4.1-mini


## JSON Schema
JSON Schema는 모델에게 아래와 같은 내용을 명확히 전달한다.
1. 어떤 key가 필요한지
2. 각 값의 자료형이 무엇인지
3. 추가 속성을 허용할지
4. 반드시 포함되어야할 필드가 무엇인지

의도하는 출력 구조
```
{
    "original" : "원문 제목",
    "corrected" : "교정된 제목",
    "issues" : [
        {
            "type" : "오류 유형",
            "before" : "수정 전 표현",
            "after" : "수정 후 표현",
            "reason" : "수정 이유",
            
        }
    ]
}
```

In [6]:
headline_schema = {
    "type" : "json_schema",
    "name" : "headline_correction",
    "strict" : True,
    "schema" : {
        # 최종 응답은 json객체
        "type" : "object",
        # properties에 정의하지 않은 key는 허용하지 않음
        "additionalProperties" : False,
        "properties" : {
            "original" : {"type":"string"},
            "corrected" : {"type":"string"},
            "issues" : {
                "type" : "array",
                "items" : {
                    "type" : "object",
                    "additionalProperties" : False,
                    "properties" : {
                        "type" : {"type": "string"},
                        "before" : {"type": "string"},
                        "after" : {"type": "string"},
                        "reason" : {"type": "string"},
                    },
                    "required" : ["type","before","after","reason"]
                }
            }
        },
        "required" : ["original","corrected","issues"]
    }
}

In [7]:
import json

headline = "AI 산업 빠르게 성장중... 기업들 투자 늘린다"

response = client.responses.create(
    model=DEFAULT_MODEL,
    instructions="너는 뉴스 제목을 교정하는 편집자이다. 맞춤법, 뛰어쓰기, 어조를 점검한다.",
    input=f"다음 제목을 교정해줘:{headline}",
    text={"format":headline_schema}
)

result = json.loads(response.output_text)
result

{'original': 'AI 산업 빠르게 성장중... 기업들 투자 늘린다',
 'corrected': 'AI 산업 빠르게 성장 중... 기업들 투자 늘린다',
 'issues': [{'type': '띄어쓰기',
   'before': '성장중',
   'after': '성장 중',
   'reason': "복합 동사 '성장 중'은 띄어 써야 합니다."}]}